# Homework assignment



> Oskar Krawczyk, AIML22 Group 3





---

As part of this homework assignemnt, we will go through topics covered in week 2 of Lady Margaret Hall Summer Programme, specifically computer vision for object classification using pretrained transformers: ViT-B/16 (Vision Transformer Base with 16x16 patch size) and Swin (Shifted Window Transformer). To do so, we will create a custom dataset based on the [Cell type dataset](https://www.kaggle.com/competitions/cell-type-prediction/data). Then, we will consider a model based on the combination of ViT and Swin Transformers, train it on the custom dataset and eventually evaluate its results using various image classification metrics - in particular accuracy.

In the following notebook, I split my solution into six separate sections - Part 1, ..., 6 - which follow the convention used in the initial [ViT_Swin_SlidingWindow_Cells_Classification_Homework_Assignment.ipynb](https://www.dropbox.com/scl/fi/v1fxjy1dktpf7990u2gy6/ViT_Swin_SlidingWindow_Cells_Classification_Homework_Assignment.ipynb?rlkey=4al6imarpkykxs671bm3fl8g3&st=vxjmvkkj&e=1&dl=0) notebook. Each section includes the code as part of my solution as well as a brief explanation of certain concepts not mentioned during lectures/seminars. At the end of this notebook, I also denote resources used to complete and/or improve the final results and additional steps I have taken (beyond the required ones) to complete the notebook.

---



## Part 1

In this section we first import all necessary libraries into a single cell. The reason is that in the future if someone tries to access the very same code, it is preferable to keep it in the single cell instead of having imports scattered across the entire notebook. Then, we set up the notebook by specifying settings for plotting (matplotlib/seaborn), device and verify versions of libraries imported. Finally, we download the dataset, process it and, at the end, we explore its structure further (number of classes, samples, etc.)

In [9]:
# !pip install -qq kagglehub # Run if kagglehub is not installed by default

#### Library Imports

In [10]:
import os
import kagglehub
import random
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

#### Notebook setup

Here, we configure the default settings for the notebook. First, we set the random seed to ensure the reproducibility of the results. Then, based on my preferences, we configure the default settings for matplotlib and seaborn packages to ensure consistent plotting. Finally, we display information regarding the imported libraries - if any methods change in the future versions of these packages, this is a reference point.

In [11]:
random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)

plt.rcParams["figure.figsize"] = (8, 8)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = False
plt.rcParams["figure.dpi"] = 100

sns.set_theme(style = "darkgrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("--- Notebook setup info ---")
print(f"Device:        {device}")
print(f"PyTorch:       {torch.__version__}")
print(f"Numpy:         {np.__version__}")

--- Notebook setup info ---
Device:        cpu
PyTorch:       2.11.0+cpu
Numpy:         2.0.2


#### Download the data

There are two options for obtaining the cell types dataset so that we can use it in this notebook and our choice is determined by what we want to achieve. If we do care about independence from the environment used - whether we are running the code on Google Colab, local Jupyter via Anaconda or remote servers - the best option is to manually download the entire dataset. This way, regardless of where we are, we can store it locally and work on it directly. The second method is using cached datasets provided by Google Colab or downloading the dataset via kagglehub we imported above.

Nevertheless, we need to keep in mind that on Google Colab, we will not be downloading the dataset directly into our project, but instead we will use a cache which in turn can speed up computations, whereas running the code locally will simply download the dataset, making this method almost equally universal to the first one.

**Note**: Considering both options, we shall use the second method by default, yet we leave the first one commented out so that you can use it if you want to. Using either of them should not cause any errors or problems with notebook's execution.

##### Downloading directly

In [12]:
# !kaggle datasets download -d mohammad2012191/cells-types --force --unzip -p ./cells_dataset # Run if you want to download the data regardless of the environment.

In [24]:
# dataset_path = "./cells_dataset/data.csv"
# images_dir = "./cells_dataset/images"

##### Downloading using kagglehub

In [27]:
raw_path = kagglehub.dataset_download("mohammad2012191/cells-types", force_download = True)

dataset_path = glob.glob(os.path.join(raw_path, "*.csv"))[0]
images_dir = glob.glob(os.path.join(raw_path, "images/*.png"))

Using Colab cache for faster access to the 'cells-types' dataset.


#### Process & Explore the data

In [ ]:
data = pd.read_csv(dataset_path)

In [54]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2578 entries, 0 to 2577
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         2578 non-null   int64 
 1   cell_type  2578 non-null   string
dtypes: int64(1), string(1)
memory usage: 40.4 KB
None


As we can see, the dataset itself is already free of null values. What can be noticed, however, is that the cell_type is not a string, but an object. We can refactor them easily to python strings in the following way.

In [53]:
data["cell_type"] = data["cell_type"].astype("string")

print(data.dtypes)

id                    int64
cell_type    string[python]
dtype: object


Having said that, we can now collect all available classes and create a 1:1 mapping between these cell types/labels and their corresponding indices.

In [65]:
classes = list(data["cell_type"].unique())
label2idx = {label: idx for idx, label in enumerate(classes)}

print("Mapping cell types to indices")
for item in label2idx:
    value = label2idx[item]
    print(f"{value} -> {item}")

Mapping cell types to indices
0 -> shsy5y
1 -> astro
2 -> cort


In [ ]:
data["label"] = data["cell_type"].map(label2idx)

Having the data prepared, we can now perform stratified train/test split with a ratio of 80%/20% with labels criterion.

**Note**: Stratified split is a data sampling method that divided a dataset so that the proportion of each class or category remains the same in both the training and testing subsets. [Source](https://medium.com/@becaye-balde/why-you-should-use-stratified-split-bddb6dadd34e)

We will see why it actually matters and whether it is indeed performed properly in the upcoming sections while exploring the dataset.

In [20]:
train_data, test_data = train_test_split(data, test_size = .2, stratify = data["cell_type"], random_state = 2026)

In [41]:
print("Dataset - Cell Types - has been configured successfully.\n")

print("--- Dataset information ---")
print(f"Dataset size:         {len(data)}")
print(f"Train set size:       {len(train_data)}")
print(f"Test  set size:       {len(test_data)}\n")

print("Classes in the dataset")
for item in label2idx:
    print(f"{label2idx[item]}:   {item}")

Dataset - Cell Types - has been configured successfully.

--- Dataset information ---
Dataset size:         2578
Train set size:       2062
Test  set size:       516

Classes in the dataset
0:   shsy5y
1:   astro
2:   cort


## Part 2

In this section we will go deeper into the data we downloaded in the previous section. We start by initialising data transformations that will be applied both to train and test datasets. Then, we create the CellDataset class which inherits from torch Dataset. Eventually, we transform our extracted data into the datasets and we can finally explore its structure.

#### Explore transformations

In [22]:
IMAGENET_STATS = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]

train_transforms = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*IMAGENET_STATS)
])

test_transforms = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(*IMAGENET_STATS)
])

#### Define dataset

In [23]:
class CellDataset(Dataset):
    '''
    Custom dataset class for handling Cell Types data.
    '''


    def __init__(self, data, images_dir, label2idx, transform = None):
        '''
        Initialises a new instance of the CellDataset class.

        '''

        super().__init__()
        self.data = data
        self.images_dir = images_dir
        self.label2idx = label2idx
        self.transform = transform

    def __len__(self):
        '''
        Returns the number of samples in the dataset.
        '''

        return len(self.data)

    def __getitem__(self, idx):
        '''

        '''





    def __str__(self):
        '''

        '''

        pass

## Part 3

## Part 4

## Part 5

## Part 6

## Summary

Here summary

## References



*   ViT16 documentation - https://docs.pytorch.org/vision/main/models/generated/torchvision.models.vit_b_16.html
*   SwinT paper - https://arxiv.org/pdf/2103.14030




---



logo_lg.svg